In [ ]:
import os
import sys
import json
from pyspark.sql import SparkSession

sys.path.append(os.path.abspath(".."))

from heal import CodeMaster
from dataio import DataWriter

# Start Spark Session
spark = (
    SparkSession.builder
    .appName("DeltaPipeline")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
print("Spark Started")

# Load Config Files
with open("../config/test_rules.json") as f:
    rules = json.load(f)

with open("../config/source_deltalake.json") as f:
    source = json.load(f)

print("Configs loaded")

# Read Source Data (Postgres via JDBC)
src = source["source"]
df = (
    spark.read
    .format("jdbc")
    .option("url", src["connection"]["url"])
    .option("dbtable", f"{src['schema']}.{src['table']}")
    .option("user", src["connection"]["username"])
    .option("password", src["connection"]["password"])
    .option("fetchsize", src["extraction"]["batch_size"])
    .load()
)

print("\nInput Data")
df.show(truncate=False)

# Run Validation + Healing Pipeline
pipeline = CodeMaster(
    rules_dict=rules,
    input_data=df,
    spark=spark
)
results = pipeline.run()

correct_df = results[0]
healed_df = results[1]
dead_df = results[2]

# Display Outputs
print("\n========== CORRECT ==========")
correct_df.show(truncate=False)

print("\n========== HEALED ==========")
healed_df.show(truncate=False)

print("\n========== DEAD RECORDS ==========")
dead_df.show(truncate=False)

# Write Outputs
writer = DataWriter(
    config=source,
    spark=spark
)

writer.data_write(correct_df, "correct_delta")
writer.data_write(healed_df, "healed_delta")
writer.data_write(dead_df, "dead_records")

print("\nFiles written successfully")

# Verify Saved Delta Output
from pyspark.sql.utils import AnalysisException
try:
    print("\nVerifying saved correct_delta table:")
    saved_correct = spark.read.format("delta").load("../output/correct_delta")
    saved_correct.show(truncate=False)

    print("\nVerifying saved healed_delta table:")
    saved_healed = spark.read.format("delta").load("../output/healed_delta")
    saved_healed.show(truncate=False)

    print("\nVerifying saved dead_records:")
    saved_dead = spark.read.parquet("../output/dead_records")
    saved_dead.show(truncate=False)
    
    print("\nVerifying saved dead_records:")

except AnalysisException:
    print("No dead records — all rows passed validation or were healed.")

# Stop Spark Session
spark.stop()
print("\nPipeline Completed")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 00:48:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Started
Configs loaded

Input Data


2026-05-24 00:48:26,328 - INFO - HealData initialized with status columns
2026-05-24 00:48:26,328 - INFO - Pipeline started
2026-05-24 00:48:26,329 - INFO - Applying rule rule_001 - order_id_not_null
2026-05-24 00:48:26,329 - INFO - [rule_001] Applying not_null check on field: 'order_id'
2026-05-24 00:48:26,352 - INFO - [rule_001] Null rows on 'order_id' marked as dead
2026-05-24 00:48:26,352 - INFO - Applying rule rule_002 - customer_id_not_null
2026-05-24 00:48:26,352 - INFO - [rule_002] Applying not_null check on field: 'customer_id'
2026-05-24 00:48:26,368 - INFO - [rule_002] Null rows on 'customer_id' marked as dead
2026-05-24 00:48:26,368 - INFO - Applying rule rule_003 - amount_range
2026-05-24 00:48:26,368 - INFO - [rule_003] Applying range validation on field 'amount' between 0.01 and 999999.99
2026-05-24 00:48:26,406 - INFO - [rule_003] Range failures on 'amount' healed via clamp
2026-05-24 00:48:26,406 - INFO - Applying rule rule_004 - email_regex
2026-05-24 00:48:26,406 - I

+--------+-----------+------------------+-----------+---------+-------------------+
|order_id|customer_id|customer_email    |amount     |status   |created_at         |
+--------+-----------+------------------+-----------+---------+-------------------+
|1001    |C001       |john@example.com  |250.50     |completed|2026-05-20 10:30:00|
|1002    |C002       |sarah@gmail.com   |1500.00    |pending  |2026-05-21 09:00:00|
|1003    |C003       |mike@yahoo.com    |99.99      |completed|2026-05-18 14:20:00|
|1004    |C004       |invalid_email     |450.25     |completed|2026-05-22 11:00:00|
|1005    |C005       |anna@gmail.com    |-500.00    |completed|2026-05-19 08:00:00|
|1006    |C006       |bob@gmail         |100.00     |completed|2026-05-23 12:00:00|
|1007    |NULL       |jenny@gmail.com   |700.00     |pending  |2026-05-20 15:00:00|
|NULL    |C008       |mark@gmail.com    |800.00     |completed|2026-05-21 16:00:00|
|1008    |C009       |alex@gmail.com    |10000000.00|completed|2026-05-20 09

2026-05-24 00:48:26,859 - INFO - [rule_006] Deduplication completed on key 'order_id'
2026-05-24 00:48:26,871 - INFO - Pipeline completed



========== CORRECT ==========
+--------+-----------+----------------+---------+---------+-------------------+-------+---------+
|order_id|customer_id|customer_email  |amount   |status   |created_at         |_status|_heal_log|
+--------+-----------+----------------+---------+---------+-------------------+-------+---------+
|1001    |C001       |john@example.com|250.5    |completed|2026-05-20 10:30:00|correct|NULL     |
|1002    |C002       |sarah@gmail.com |1500.0   |pending  |2026-05-21 09:00:00|correct|NULL     |
|1003    |C003       |mike@yahoo.com  |99.99    |completed|2026-05-18 14:20:00|correct|NULL     |
|1004    |C004       |NULL            |450.25   |completed|2026-05-22 11:00:00|correct|NULL     |
|1005    |C005       |anna@gmail.com  |0.01     |completed|2026-05-19 08:00:00|correct|NULL     |
|1006    |C006       |NULL            |100.0    |completed|2026-05-23 12:00:00|correct|NULL     |
|1008    |C009       |alex@gmail.com  |999999.99|completed|2026-05-20 09:00:00|correct|

26/05/24 00:48:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-05-24 00:48:29,549 - INFO - Written to Delta: ../output/correct_delta
2026-05-24 00:48:30,171 - INFO - Written to Delta: ../output/healed_delta
2026-05-24 00:48:30,317 - INFO - Written to local storage: ../output/dead_records/dead_records



Files written successfully

Verifying saved correct_delta table:
+--------+-----------+----------------+---------+---------+-------------------+-------+---------+
|order_id|customer_id|customer_email  |amount   |status   |created_at         |_status|_heal_log|
+--------+-----------+----------------+---------+---------+-------------------+-------+---------+
|1001    |C001       |john@example.com|250.5    |completed|2026-05-20 10:30:00|correct|NULL     |
|1002    |C002       |sarah@gmail.com |1500.0   |pending  |2026-05-21 09:00:00|correct|NULL     |
|1003    |C003       |mike@yahoo.com  |99.99    |completed|2026-05-18 14:20:00|correct|NULL     |
|1004    |C004       |NULL            |450.25   |completed|2026-05-22 11:00:00|correct|NULL     |
|1005    |C005       |anna@gmail.com  |0.01     |completed|2026-05-19 08:00:00|correct|NULL     |
|1006    |C006       |NULL            |100.0    |completed|2026-05-23 12:00:00|correct|NULL     |
|1008    |C009       |alex@gmail.com  |999999.99|com

In [2]:
import traceback

try:
    df = (
        spark.read
        .format("jdbc")
        .option("url", src["connection"]["url"])
        .option("dbtable", f"{src['schema']}.{src['table']}")
        .option("user", src["connection"]["username"])
        .option("password", src["connection"]["password"])
        .option("fetchsize", src["extraction"]["batch_size"])
        .load()
    )
except Exception as e:
    traceback.print_exc()

Traceback (most recent call last):
  File "/var/folders/5y/c7j8s1fs3xb_90fv3qjq24bm0000gn/T/ipykernel_13261/3107856854.py", line 12, in <module>
    .load()
     ^^^^^^
  File "/opt/homebrew/opt/apache-spark/libexec/python/pyspark/sql/readwriter.py", line 318, in load
    return self._df(self._jreader.load())
                    ^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/py4j/java_gateway.py", line 1362, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/opt/apache-spark/libexec/python/pyspark/errors/exceptions/captured.py", line 263, in deco
    return f(*a, **kw)
           ^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/py4j/protocol.py", line 327, in get_return_value
    raise Py4JJavaError(
py4j.protocol.Py4JJavaError: An error occurred while calling o48.load.
: org.postgresql.util.PSQLException: FATAL: database "orders" does not exist
	at org.postgresql.core.v3.QueryExecutorImpl.r